# Simulated Annealing para distribuci?n de combustibles

Notebook generado a partir de `simulated_annealing_combustibles.py`.


In [ ]:
import copy
import math
import random
from typing import Dict, List, Tuple, Any




## DATOS DEL PROBLEMA


In [ ]:
CAMIONES = {
    "T1": {
        "capacidad": {"C0": 8000, "C1": 7000},
        "costo_fijo": 500,
        "velocidad": 60,       # km/h
        "salida": "05:00"
    },
    "T2": {
        "capacidad": {"C0": 6000, "C1": 9000},
        "costo_fijo": 400,
        "velocidad": 60,       # km/h
        "salida": "05:30"
    }
}

DEMANDAS = {
    1: {"R": 3000, "D": 2000},
    2: {"R": 4000, "D": 1500},
    3: {"R": 2500, "D": 3000},
    4: {"R": 1000, "D": 2500}
}

VENTANAS = {
    1: ("06:00", "10:00"),
    2: ("07:00", "12:00"),
    3: ("08:00", "14:00"),
    4: ("06:00", "11:00")
}

DISTANCIAS = {
    (0, 1): 20, (0, 2): 35, (0, 3): 15, (0, 4): 40,
    (1, 0): 20, (2, 0): 35, (3, 0): 15, (4, 0): 40,

    (1, 2): 10, (2, 1): 10,
    (1, 3): 25, (3, 1): 25,
    (1, 4): 30, (4, 1): 30,

    (2, 3): 20, (3, 2): 20,
    (2, 4): 15, (4, 2): 15,

    (3, 4): 30, (4, 3): 30
}

PRODUCTOS = ["R", "D"]
COMPARTIMENTOS = ["C0", "C1"]
ESTACIONES = [1, 2, 3, 4]

COSTO_KM = 2
COSTO_SHORTAGE = 10
TIEMPO_SERVICIO = 30  # minutos
DELTA_MAX = 0.30

# Penalizaciones grandes para forzar factibilidad
LAMBDA_CAPACIDAD = 10_000
LAMBDA_ESTABILIDAD = 50_000
LAMBDA_TIEMPO = 1_000
LAMBDA_DUPLICADOS = 100_000
LAMBDA_ESTACIONES_NO_ASIGNADAS = 100_000




## UTILIDADES


In [ ]:
def hora_a_minutos(hora: str) -> int:
    """Convierte una hora HH:MM a minutos desde las 00:00."""
    h, m = hora.split(":")
    return int(h) * 60 + int(m)


def minutos_a_hora(minutos: float) -> str:
    """Convierte minutos desde las 00:00 a formato HH:MM."""
    minutos = int(round(minutos))
    h = minutos // 60
    m = minutos % 60
    return f"{h:02d}:{m:02d}"


def distancia_ruta(ruta: List[int]) -> float:
    """
    Calcula la distancia total de una ruta:
    Depósito -> estaciones -> Depósito.
    """
    if not ruta:
        return 0

    distancia = 0
    nodo_actual = 0

    for estacion in ruta:
        distancia += DISTANCIAS[(nodo_actual, estacion)]
        nodo_actual = estacion

    distancia += DISTANCIAS[(nodo_actual, 0)]
    return distancia


def estaciones_en_solucion(solucion: Dict[str, Any]) -> List[int]:
    """Retorna todas las estaciones presentes en las rutas."""
    estaciones = []
    for datos in solucion.values():
        estaciones.extend(datos["ruta"])
    return estaciones




## REPRESENTACIÓN Y SOLUCIÓN INICIAL


In [ ]:
def generar_solucion_inicial() -> Dict[str, Any]:
    """
    Solución inicial basada en la propuesta del operador:
    T1: Depósito -> 1 -> 3 -> Depósito
    T2: Depósito -> 2 -> 4 -> Depósito
    """
    return {
        "T1": {
            "ruta": [1, 3],
            "asignacion": {"C0": "R", "C1": "D"}
        },
        "T2": {
            "ruta": [2, 4],
            "asignacion": {"C0": "D", "C1": "R"}
        }
    }


def normalizar_solucion(solucion: Dict[str, Any]) -> Dict[str, Any]:
    """
    Corrige duplicados y estaciones perdidas para mantener una solución estructuralmente válida.
    No garantiza factibilidad de capacidad, estabilidad ni tiempo.
    """
    nueva = copy.deepcopy(solucion)

    vistas = set()
    duplicadas = []

    for camion in nueva:
        ruta_limpia = []
        for estacion in nueva[camion]["ruta"]:
            if estacion not in vistas:
                ruta_limpia.append(estacion)
                vistas.add(estacion)
            else:
                duplicadas.append(estacion)
        nueva[camion]["ruta"] = ruta_limpia

    faltantes = [e for e in ESTACIONES if e not in vistas]

    for estacion in faltantes:
        camion = random.choice(list(nueva.keys()))
        pos = random.randint(0, len(nueva[camion]["ruta"]))
        nueva[camion]["ruta"].insert(pos, estacion)

    return nueva




## CÁLCULO DE ENTREGAS Y CARGAS


In [ ]:
def calcular_demanda_por_producto(ruta: List[int]) -> Dict[str, float]:
    """Calcula la demanda total Regular y Diésel de una ruta."""
    total = {"R": 0, "D": 0}

    for estacion in ruta:
        total["R"] += DEMANDAS[estacion]["R"]
        total["D"] += DEMANDAS[estacion]["D"]

    return total


def asignar_cargas_a_compartimentos(solucion: Dict[str, Any]) -> Tuple[Dict[str, Dict[str, float]], float]:
    """
    Calcula la carga inicial de cada compartimento según la ruta y asignación.

    Si un producto no tiene compartimento asignado en un camión que visita estaciones
    con demanda de ese producto, esa demanda se considera shortage.

    Si un producto tiene más de un compartimento asignado, se distribuye de manera
    simple usando la capacidad disponible.
    """
    cargas_por_camion = {}
    shortage = 0

    for camion, datos in solucion.items():
        ruta = datos["ruta"]
        asignacion = datos["asignacion"]
        capacidades = CAMIONES[camion]["capacidad"]

        cargas = {"C0": 0, "C1": 0}
        demanda_producto = calcular_demanda_por_producto(ruta)

        for producto in PRODUCTOS:
            compartimentos_producto = [
                c for c in COMPARTIMENTOS
                if asignacion[c] == producto
            ]

            demanda = demanda_producto[producto]

            if demanda == 0:
                continue

            if not compartimentos_producto:
                shortage += demanda
                continue

            restante = demanda

            for c in compartimentos_producto:
                espacio = capacidades[c] - cargas[c]
                asignado = min(restante, espacio)
                cargas[c] += asignado
                restante -= asignado

            if restante > 0:
                # No alcanzó la capacidad para cubrir todo el producto
                shortage += restante

        cargas_por_camion[camion] = cargas

    return cargas_por_camion, shortage


def producto_de_compartimento(solucion: Dict[str, Any], camion: str, compartimento: str) -> str:
    return solucion[camion]["asignacion"][compartimento]




## COSTOS


In [ ]:
def costo_distancia(solucion: Dict[str, Any]) -> float:
    total = 0

    for camion, datos in solucion.items():
        ruta = datos["ruta"]
        total += distancia_ruta(ruta) * COSTO_KM

    return total


def costo_fijo(solucion: Dict[str, Any]) -> float:
    total = 0

    for camion, datos in solucion.items():
        if datos["ruta"]:
            total += CAMIONES[camion]["costo_fijo"]

    return total


def costo_shortage(shortage_litros: float) -> float:
    return shortage_litros * COSTO_SHORTAGE




## PENALIZACIONES DE FACTIBILIDAD


In [ ]:
def penalizacion_capacidad(solucion: Dict[str, Any]) -> float:
    """
    Penaliza excesos de capacidad.
    En la función de carga ya se intenta no superar capacidad, por lo que esta
    penalización normalmente será cero. Se mantiene para robustez.
    """
    cargas, _ = asignar_cargas_a_compartimentos(solucion)
    exceso = 0

    for camion in solucion:
        for c in COMPARTIMENTOS:
            capacidad = CAMIONES[camion]["capacidad"][c]
            if cargas[camion][c] > capacidad:
                exceso += cargas[camion][c] - capacidad

    return exceso


def penalizacion_estabilidad(solucion: Dict[str, Any]) -> float:
    """
    Penaliza diferencias de fill ratio mayores a DELTA_MAX.
    Se evalúa al salir del depósito y después de cada entrega.
    """
    cargas_iniciales, _ = asignar_cargas_a_compartimentos(solucion)
    penalizacion = 0

    for camion, datos in solucion.items():
        ruta = datos["ruta"]
        asignacion = datos["asignacion"]
        capacidades = CAMIONES[camion]["capacidad"]

        cargas = copy.deepcopy(cargas_iniciales[camion])

        def diferencia_fill_ratio() -> float:
            fr_c0 = cargas["C0"] / capacidades["C0"]
            fr_c1 = cargas["C1"] / capacidades["C1"]
            return abs(fr_c0 - fr_c1)

        # Estado inicial
        dif = diferencia_fill_ratio()
        if dif > DELTA_MAX:
            penalizacion += dif - DELTA_MAX

        # Estados después de cada estación
        for estacion in ruta:
            for c in COMPARTIMENTOS:
                producto = asignacion[c]
                demanda_producto = DEMANDAS[estacion][producto]

                entrega = min(cargas[c], demanda_producto)
                cargas[c] -= entrega

            dif = diferencia_fill_ratio()
            if dif > DELTA_MAX:
                penalizacion += dif - DELTA_MAX

    return penalizacion


def penalizacion_ventanas_tiempo(solucion: Dict[str, Any]) -> float:
    """
    Penaliza llegadas fuera de ventana de tiempo.
    Si llega antes, se permite esperar, por lo que no se penaliza llegar temprano.
    Se penaliza solo llegar después del cierre de ventana.
    """
    penalizacion = 0

    for camion, datos in solucion.items():
        ruta = datos["ruta"]

        if not ruta:
            continue

        velocidad = CAMIONES[camion]["velocidad"]
        tiempo_actual = hora_a_minutos(CAMIONES[camion]["salida"])
        nodo_actual = 0

        for estacion in ruta:
            tiempo_viaje = (DISTANCIAS[(nodo_actual, estacion)] / velocidad) * 60
            llegada = tiempo_actual + tiempo_viaje

            inicio = hora_a_minutos(VENTANAS[estacion][0])
            fin = hora_a_minutos(VENTANAS[estacion][1])

            if llegada > fin:
                penalizacion += llegada - fin

            # Si llega antes, espera hasta que abra la ventana
            inicio_servicio = max(llegada, inicio)
            tiempo_actual = inicio_servicio + TIEMPO_SERVICIO
            nodo_actual = estacion

    return penalizacion


def penalizacion_estaciones(solucion: Dict[str, Any]) -> float:
    """
    Penaliza estaciones duplicadas o no asignadas.
    """
    estaciones = estaciones_en_solucion(solucion)

    faltantes = len(set(ESTACIONES) - set(estaciones))
    duplicados = len(estaciones) - len(set(estaciones))

    return faltantes + duplicados




## EVALUACIÓN TOTAL


In [ ]:
def evaluar_solucion(solucion: Dict[str, Any]) -> Dict[str, float]:
    """
    Retorna desglose completo de costos y penalizaciones.
    """
    cargas, shortage_litros = asignar_cargas_a_compartimentos(solucion)

    c_dist = costo_distancia(solucion)
    c_fijo = costo_fijo(solucion)
    c_short = costo_shortage(shortage_litros)

    p_cap = penalizacion_capacidad(solucion)
    p_est = penalizacion_estabilidad(solucion)
    p_time = penalizacion_ventanas_tiempo(solucion)
    p_estaciones = penalizacion_estaciones(solucion)

    penalizacion_total = (
        LAMBDA_CAPACIDAD * p_cap +
        LAMBDA_ESTABILIDAD * p_est +
        LAMBDA_TIEMPO * p_time +
        LAMBDA_ESTACIONES_NO_ASIGNADAS * p_estaciones
    )

    costo_total = c_dist + c_fijo + c_short + penalizacion_total

    return {
        "costo_total": costo_total,
        "costo_distancia": c_dist,
        "costo_fijo": c_fijo,
        "shortage_litros": shortage_litros,
        "costo_shortage": c_short,
        "penalizacion_capacidad": p_cap,
        "penalizacion_estabilidad": p_est,
        "penalizacion_tiempo": p_time,
        "penalizacion_estaciones": p_estaciones,
        "penalizacion_total": penalizacion_total
    }




## GENERACIÓN DE VECINOS


In [ ]:
def generar_vecino(solucion: Dict[str, Any]) -> Dict[str, Any]:
    """
    Genera una solución vecina usando uno de cuatro movimientos:
    1. swap: intercambiar estaciones entre camiones.
    2. move: mover estación de un camión a otro.
    3. reverse: invertir ruta de un camión.
    4. asignacion: intercambiar productos entre C0 y C1.
    """
    vecino = copy.deepcopy(solucion)
    movimiento = random.choice(["swap", "move", "reverse", "asignacion"])

    camiones = list(vecino.keys())

    if movimiento == "swap":
        c1, c2 = random.sample(camiones, 2)

        if vecino[c1]["ruta"] and vecino[c2]["ruta"]:
            i = random.randrange(len(vecino[c1]["ruta"]))
            j = random.randrange(len(vecino[c2]["ruta"]))

            vecino[c1]["ruta"][i], vecino[c2]["ruta"][j] = (
                vecino[c2]["ruta"][j],
                vecino[c1]["ruta"][i]
            )

    elif movimiento == "move":
        c1, c2 = random.sample(camiones, 2)

        if vecino[c1]["ruta"]:
            i = random.randrange(len(vecino[c1]["ruta"]))
            estacion = vecino[c1]["ruta"].pop(i)

            pos = random.randint(0, len(vecino[c2]["ruta"]))
            vecino[c2]["ruta"].insert(pos, estacion)

    elif movimiento == "reverse":
        camion = random.choice(camiones)

        if len(vecino[camion]["ruta"]) >= 2:
            vecino[camion]["ruta"].reverse()

    elif movimiento == "asignacion":
        camion = random.choice(camiones)

        vecino[camion]["asignacion"]["C0"], vecino[camion]["asignacion"]["C1"] = (
            vecino[camion]["asignacion"]["C1"],
            vecino[camion]["asignacion"]["C0"]
        )

    return normalizar_solucion(vecino)




## SIMULATED ANNEALING


In [ ]:
def simulated_annealing(
    solucion_inicial: Dict[str, Any],
    temperatura_inicial: float = 1000,
    temperatura_final: float = 0.01,
    alpha: float = 0.95,
    iteraciones_por_temperatura: int = 100,
    semilla: int = 42
) -> Tuple[Dict[str, Any], Dict[str, float]]:
    """
    Ejecuta Simulated Annealing y retorna la mejor solución encontrada.
    """
    random.seed(semilla)

    solucion_actual = copy.deepcopy(solucion_inicial)
    solucion_actual = normalizar_solucion(solucion_actual)

    eval_actual = evaluar_solucion(solucion_actual)
    costo_actual = eval_actual["costo_total"]

    mejor_solucion = copy.deepcopy(solucion_actual)
    mejor_eval = copy.deepcopy(eval_actual)
    mejor_costo = costo_actual

    T = temperatura_inicial

    while T > temperatura_final:
        for _ in range(iteraciones_por_temperatura):
            vecino = generar_vecino(solucion_actual)
            eval_vecino = evaluar_solucion(vecino)
            costo_vecino = eval_vecino["costo_total"]

            delta = costo_vecino - costo_actual

            if delta < 0:
                solucion_actual = vecino
                eval_actual = eval_vecino
                costo_actual = costo_vecino
            else:
                probabilidad = math.exp(-delta / T)
                if random.random() < probabilidad:
                    solucion_actual = vecino
                    eval_actual = eval_vecino
                    costo_actual = costo_vecino

            if costo_actual < mejor_costo:
                mejor_solucion = copy.deepcopy(solucion_actual)
                mejor_eval = copy.deepcopy(eval_actual)
                mejor_costo = costo_actual

        T *= alpha

    return mejor_solucion, mejor_eval




## REPORTE DE SOLUCIÓN


In [ ]:
def calcular_llegadas(solucion: Dict[str, Any]) -> Dict[str, List[Tuple[int, str]]]:
    """
    Calcula las horas de llegada por camión.
    """
    llegadas = {}

    for camion, datos in solucion.items():
        ruta = datos["ruta"]
        velocidad = CAMIONES[camion]["velocidad"]

        tiempo_actual = hora_a_minutos(CAMIONES[camion]["salida"])
        nodo_actual = 0

        llegadas[camion] = []

        for estacion in ruta:
            tiempo_viaje = (DISTANCIAS[(nodo_actual, estacion)] / velocidad) * 60
            llegada = tiempo_actual + tiempo_viaje

            inicio = hora_a_minutos(VENTANAS[estacion][0])
            inicio_servicio = max(llegada, inicio)

            llegadas[camion].append((estacion, minutos_a_hora(llegada)))

            tiempo_actual = inicio_servicio + TIEMPO_SERVICIO
            nodo_actual = estacion

    return llegadas


def imprimir_solucion(solucion: Dict[str, Any], evaluacion: Dict[str, float]) -> None:
    """
    Imprime la solución de forma legible.
    """
    print("\n==============================")
    print("MEJOR SOLUCIÓN ENCONTRADA")
    print("==============================")

    cargas, shortage = asignar_cargas_a_compartimentos(solucion)
    llegadas = calcular_llegadas(solucion)

    for camion, datos in solucion.items():
        ruta = datos["ruta"]
        asignacion = datos["asignacion"]

        print(f"\nCamión {camion}")
        print(f"Ruta: Depósito -> {' -> '.join(map(str, ruta))} -> Depósito")
        print(f"Distancia: {distancia_ruta(ruta)} km")
        print(f"Asignación compartimentos: {asignacion}")
        print(f"Cargas iniciales: {cargas[camion]}")
        print("Llegadas:")

        for estacion, hora in llegadas[camion]:
            print(f"  Estación {estacion}: {hora} | Ventana {VENTANAS[estacion]}")

    print("\n------------------------------")
    print("DESGLOSE DE COSTOS")
    print("------------------------------")
    for clave, valor in evaluacion.items():
        print(f"{clave}: {valor:.2f}")




## EJECUCIÓN PRINCIPAL


In [ ]:
solucion_inicial = generar_solucion_inicial()

mejor_solucion, mejor_eval = simulated_annealing(
    solucion_inicial=solucion_inicial,
    temperatura_inicial=1000,
    temperatura_final=0.01,
    alpha=0.95,
    iteraciones_por_temperatura=100,
    semilla=42
)

imprimir_solucion(mejor_solucion, mejor_eval)

# Tabla de costos visible en el notebook
try:
    import pandas as pd
    display(pd.DataFrame(mejor_eval.items(), columns=["concepto", "valor"]))
except ImportError:
    mejor_eval
